<a href="https://colab.research.google.com/github/cuctuyetaz258/Cross-Domain-Video-Highlight-Agent/blob/main/week1/5_layer_feature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q librosa soundfile sentence-transformers scikit-learn pandas numpy yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.7 MB/s eta 0:00:00


## Phần 1.1 — Khởi tạo cấu trúc mô hình 5 tầng đặc trưng

In [2]:
import os
import json
import numpy as np
import pandas as pd
import librosa
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

# khai bao 5 tang dac trung
LAYERS = [
    "acoustic",        # Am hoc
    "paralinguistic",  # Can ngon ngu
    "linguistic",      # Ngon ngu
    "structural",      # Cau truc
    "interaction",      # Tuong tac
]

@dataclass
class FeatureWindow:
    """Mot cua so thoi gian (mac dinh 30s) chua diem so cua 5 tang dac trung."""
    start: float
    end: float
    scores: Dict[str, float] = field(default_factory=lambda: {l: 0.0 for l in LAYERS})
    meta: Dict = field(default_factory=dict)

    def total_score(self, weights: Optional[Dict[str, float]] = None) -> float:
        w = weights or {l: 1.0 / len(LAYERS) for l in LAYERS}
        return sum(self.scores.get(l, 0.0) * w.get(l, 0.0) for l in LAYERS)


class LayerExtractor:
    """Base class cho tung tang dac trung -- moi tang cu the se override extract()."""
    name: str = "base"

    def extract(self, window_data: dict) -> float:
        raise NotImplementedError


class AcousticExtractor(LayerExtractor):
    name = "acoustic"
    # TODO (Khanh Van, buoi 2): RMS energy, pitch variation, silence duration (librosa)


class ParalinguisticExtractor(LayerExtractor):
    name = "paralinguistic"
    # TODO: laughter detection, emotion (wav2vec2 / distilbert)


class LinguisticExtractor(LayerExtractor):
    name = "linguistic"
    # -> Semantic score + keyword importance, xem Phan 2 ben duoi (Tat Nguyen)


class StructuralExtractor(LayerExtractor):
    name = "structural"
    # TODO: scene change, slide transition (OpenCV / PySceneDetect)


class InteractionExtractor(LayerExtractor):
    name = "interaction"
    # -> Speaker changes, turn-taking rate, xem Phan 2 ben duoi (Tat Nguyen)


EXTRACTORS = {
    "acoustic": AcousticExtractor(),
    "paralinguistic": ParalinguisticExtractor(),
    "linguistic": LinguisticExtractor(),
    "structural": StructuralExtractor(),
    "interaction": InteractionExtractor(),
}

print("Khung 5 tang dac trung da san sang:", LAYERS)

Khung 5 tang dac trung da san sang: ['acoustic', 'paralinguistic', 'linguistic', 'structural', 'interaction']


In [3]:
# cau hinh librosa

SAMPLE_RATE = 16000       # chuan hoa 16kHz mono (khop voi ffmpeg pipeline cua nhom)
WINDOW_SEC = 30.0         # kich thuoc cua so truot (theo de cuong)
HOP_SEC = 15.0            # buoc truot (50% overlap)


def load_audio(path: str, sr: int = SAMPLE_RATE) -> Tuple[np.ndarray, int]:
    """Load 1 file audio (.wav/.mp3) ve mono, sample rate chuan."""
    y, sr = librosa.load(path, sr=sr, mono=True)
    return y, sr


def make_sliding_windows(duration_sec: float, window_sec: float = WINDOW_SEC,
                          hop_sec: float = HOP_SEC) -> List[FeatureWindow]:
    """Sinh danh sach cua so thoi gian truot tren toan bo audio/video."""
    windows = []
    t = 0.0
    while t < duration_sec:
        end = min(t + window_sec, duration_sec)
        windows.append(FeatureWindow(start=t, end=end))
        if end >= duration_sec:
            break
        t += hop_sec
    return windows


def audio_segment(y: np.ndarray, sr: int, start: float, end: float) -> np.ndarray:
    """Cat doan audio theo thoi gian (giay)."""
    return y[int(start * sr): int(end * sr)]


def run_pipeline(y: np.ndarray, sr: int, extractors: Dict[str, LayerExtractor] = None) -> pd.DataFrame:
    """Chay toan bo 5 tang tren tung cua so, tra ve DataFrame diem so theo thoi gian."""
    extractors = extractors or EXTRACTORS
    duration = len(y) / sr
    windows = make_sliding_windows(duration)
    rows = []
    for w in windows:
        seg = audio_segment(y, sr, w.start, w.end)
        row = {"start": w.start, "end": w.end}
        for name, extractor in extractors.items():
            try:
                row[name] = extractor.extract({"audio": seg, "sr": sr})
            except NotImplementedError:
                row[name] = None
        rows.append(row)
    return pd.DataFrame(rows)


print(f"Sliding window config: window={WINDOW_SEC}s, hop={HOP_SEC}s, sr={SAMPLE_RATE}Hz")

Sliding window config: window=30.0s, hop=15.0s, sr=16000Hz


In [6]:
# thu thap data
import subprocess

SAMPLE_DIR = "sample_dataset"
os.makedirs(SAMPLE_DIR, exist_ok=True)

# Dien URL YouTube mau cho tung mien (thay bang video that cua nhom: 5-30 phut,
# theo de cuong can 10-16 video cho 2 mien Lecture + Podcast)
SAMPLE_VIDEOS = {
    "lecture_01": "https://www.youtube.com/watch?v=XXXXXXXXXXX",
    "podcast_01": "https://www.youtube.com/watch?v=YYYYYYYYYYY",
}


def download_sample(name: str, url: str, out_dir: str = SAMPLE_DIR):
    """Tai audio (.wav 16kHz mono) + phu de tu dong (neu co) bang yt-dlp."""
    out_path = os.path.join(out_dir, name)
    cmd = [
        "yt-dlp", "-x", "--audio-format", "wav",
        "--postprocessor-args", f"-ar {SAMPLE_RATE} -ac 1",
        "--write-auto-sub", "--sub-lang", "en,vi",
        "-o", f"{out_path}.%(ext)s",
        url,
    ]
    subprocess.run(cmd, check=False)
    print(f"Da tai xong: {name}")


print("Khung thu thap du lieu mau da san sang.")
print("-> Dien URL that vao SAMPLE_VIDEOS roi bo comment vong for de tai.")

Khung thu thap du lieu mau da san sang.
-> Dien URL that vao SAMPLE_VIDEOS roi bo comment vong for de tai.


In [7]:
# sematic score tu transcript
from sentence_transformers import SentenceTransformer, util

semantic_model = SentenceTransformer("all-MiniLM-L6-v2")


def compute_semantic_scores(segments: List[str]) -> List[float]:
    """
    Tinh Semantic Score cho tung doan transcript bang Sentence-Transformers.
    Cach lam: embed tung cau/doan + embed vector ngu canh chung (trung binh toan bo),
    diem so = cosine similarity giua doan do va ngu canh chung
    -> doan cang 'trung tam / dai dien' cho noi dung thi diem cang cao.
    """
    if not segments:
        return []
    seg_embeddings = semantic_model.encode(segments, convert_to_tensor=True)
    doc_embedding = seg_embeddings.mean(dim=0, keepdim=True)
    sims = util.cos_sim(seg_embeddings, doc_embedding).squeeze(1)
    scores = sims.cpu().numpy().tolist()
    # normalize ve [0, 1]
    lo, hi = min(scores), max(scores)
    if hi - lo > 1e-9:
        scores = [(s - lo) / (hi - lo) for s in scores]
    return scores

# phuong an thay the dung LLm
def compute_semantic_score_llm(segments: List[str], api_call_fn) -> List[float]:
    scores = []
    for seg in segments:
        prompt = (
            f'"{seg}"'
        )
        response = api_call_fn(prompt)
        try:
            scores.append(float(response.strip()))
        except ValueError:
            scores.append(0.5)  # fallback neu LLM tra loi khong parse duoc
    return scores


print("Semantic Score module (Sentence-Transformers + LLM fallback) da san sang.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Semantic Score module (Sentence-Transformers + LLM fallback) da san sang.


In [9]:

from sklearn.feature_extraction.text import TfidfVectorizer


def compute_keyword_importance(segments: List[str], top_k: int = 5) -> List[dict]:
    """
    Tinh diem quan trong tu khoa cho tung doan bang TF-IDF tren toan bo transcript.
    Tra ve: diem tong hop (trung binh TF-IDF cua top-k tu) + danh sach tu khoa noi bat.
    """
    if not segments:
        return []
    vectorizer = TfidfVectorizer(stop_words="english", max_features=500)
    tfidf_matrix = vectorizer.fit_transform(segments)
    feature_names = np.array(vectorizer.get_feature_names_out())

    results = []
    for i in range(len(segments)):
        row = tfidf_matrix[i].toarray().flatten()
        top_idx = row.argsort()[::-1][:top_k]
        top_idx = [idx for idx in top_idx if row[idx] > 0]
        keywords = feature_names[top_idx].tolist() if len(top_idx) else []
        keyword_score = float(row[top_idx].mean()) if len(top_idx) else 0.0
        results.append({"keyword_score": keyword_score, "top_keywords": keywords})
    return results


print("Keyword Importance module (TF-IDF) da san sang.")
print("Luu y: transcript Tieng Viet nen dung stop_words rieng thay vi \'english\' - co the thay bang danh sach stopwords VN.")

Keyword Importance module (TF-IDF) da san sang.
Luu y: transcript Tieng Viet nen dung stop_words rieng thay vi 'english' - co the thay bang danh sach stopwords VN.


In [10]:
#  speaker change

@dataclass
class TranscriptTurn:
    speaker: str
    start: float
    end: float
    text: str = ""


def count_speaker_changes(turns: List[TranscriptTurn], window_start: float, window_end: float) -> dict:
    """
    Voi danh sach cac luot noi (lay tu pyannote/speaker-diarization-3.1) trong 1 cua so thoi gian:
    - Dem so lan doi speaker (turn changes)
    - Tinh turn-taking rate = so luot noi / phut
    """
    in_window = [t for t in turns if t.end > window_start and t.start < window_end]
    in_window.sort(key=lambda t: t.start)

    changes = 0
    for i in range(1, len(in_window)):
        if in_window[i].speaker != in_window[i - 1].speaker:
            changes += 1

    duration_min = max((window_end - window_start) / 60.0, 1e-6)
    turn_taking_rate = len(in_window) / duration_min  # so luot noi / phut

    return {
        "num_turns": len(in_window),
        "speaker_changes": changes,
        "turn_taking_rate_per_min": turn_taking_rate,
        "unique_speakers": len(set(t.speaker for t in in_window)),
    }


def interaction_score_from_turns(turns: List[TranscriptTurn], window_start: float,
                                  window_end: float, max_rate_ref: float = 12.0) -> float:
    """
    Chuan hoa turn-taking rate ve [0,1] lam diem Interaction Score cho tang 'Tuong tac'.
    max_rate_ref: nguong tham chieu (dieu chinh sau khi co du lieu podcast that).
    """
    stats = count_speaker_changes(turns, window_start, window_end)
    return min(stats["turn_taking_rate_per_min"] / max_rate_ref, 1.0)


print("Speaker turn-taking module da san sang.")

Speaker turn-taking module da san sang.


In [11]:
# demo voi dl gia lap

demo_segments = [
    "Chao mung cac ban den voi podcast hom nay, chung ta se noi ve AI.",
    "Day la mot khoanh khac rat quan trong trong lich su phat trien machine learning.",
    "A thi, um, toi nghi la, khong co gi dac biet lam dau.",
    "Diem mau chot o day la mo hinh Transformer da thay doi hoan toan cach chung ta xu ly ngon ngu.",
]

semantic_scores = compute_semantic_scores(demo_segments)
keyword_info = compute_keyword_importance(demo_segments)

demo_turns = [
    TranscriptTurn("A", 0, 8, demo_segments[0]),
    TranscriptTurn("B", 8, 20, demo_segments[1]),
    TranscriptTurn("A", 20, 25, demo_segments[2]),
    TranscriptTurn("B", 25, 40, demo_segments[3]),
]
interaction_stats = count_speaker_changes(demo_turns, 0, 40)

demo_df = pd.DataFrame({
    "segment": demo_segments,
    "semantic_score": semantic_scores,
    "keyword_score": [k["keyword_score"] for k in keyword_info],
    "top_keywords": [k["top_keywords"] for k in keyword_info],
})

display(demo_df)

print(interaction_stats)

print(interaction_score_from_turns(demo_turns, 0, 40))

,segment,semantic_score,keyword_score,top_keywords
0,"Chao mung cac ban den voi podcast hom nay, chu...",0.515653,0.264970,"[voi, ve, se, podcast, noi]"
1,Day la mot khoanh khac rat quan trong trong li...,0.000000,0.299728,"[trong, trien, rat, quan, phat]"
2,"A thi, um, toi nghi la, khong co gi dac biet l...",1.000000,0.309976,"[um, thi, toi, nghi, dau]"
3,Diem mau chot o day la mo hinh Transformer da ...,0.804414,0.233940,"[xu, transformer, toan, ngon, ngu]"


{'num_turns': 4, 'speaker_changes': 3, 'turn_taking_rate_per_min': 6.0, 'unique_speakers': 2}
0.5
